<a href="https://colab.research.google.com/github/aparna-2001/nifty50-stock-price-forecasting/blob/main/nifty50_monthly_eda_prediction_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import matplotlib.pyplot as plt

In [31]:
nifty_df = pd.read_csv('nifty50_25years_ohlcv_1999_2026.csv')

In [32]:
nifty_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6286 entries, 0 to 6285
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    6286 non-null   object 
 1   Open    6286 non-null   float64
 2   High    6286 non-null   float64
 3   Low     6286 non-null   float64
 4   Close   6286 non-null   float64
 5   Volume  6286 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 294.8+ KB


* Aggreggating it to monthly data

In [33]:
nifty_df = pd.read_csv('nifty50_25years_ohlcv_1999_2026.csv', parse_dates=['Date'])

In [34]:
df = pd.read_csv('nifty50_25years_ohlcv_1999_2026.csv', parse_dates=['Date'])
df.set_index('Date', inplace=True)

monthly = df.resample('MS').agg(
    Month_Open  = ('Open',  'first'),
    Month_Close = ('Close', 'last'),
    Month_High  = ('High',  'max'),
    Month_Low   = ('Low',   'min'),
    Avg_Volume  = ('Volume','mean')
).reset_index()

monthly.rename(columns={'Date': 'Month'}, inplace=True)

# Check which months are missing
print(monthly[monthly['Month_Close'].isnull()][['Month']])

# Drop missing months
monthly.dropna(inplace=True)
print(monthly.shape)
print(monthly.dtypes)

         Month
49  2003-02-01
50  2003-03-01
51  2003-04-01
52  2003-05-01
53  2003-06-01
54  2003-07-01
55  2003-08-01
56  2003-09-01
57  2003-10-01
58  2003-11-01
59  2003-12-01
96  2007-01-01
97  2007-02-01
98  2007-03-01
99  2007-04-01
100 2007-05-01
101 2007-06-01
102 2007-07-01
103 2007-08-01
(308, 6)
Month          datetime64[ns]
Month_Open            float64
Month_Close           float64
Month_High            float64
Month_Low             float64
Avg_Volume            float64
dtype: object


In [35]:
monthly.info()

<class 'pandas.core.frame.DataFrame'>
Index: 308 entries, 0 to 326
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Month        308 non-null    datetime64[ns]
 1   Month_Open   308 non-null    float64       
 2   Month_Close  308 non-null    float64       
 3   Month_High   308 non-null    float64       
 4   Month_Low    308 non-null    float64       
 5   Avg_Volume   308 non-null    float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 16.8 KB


*feature engineering*

In [36]:
monthly['Monthly_Return_%']  = (monthly['Month_Close'] - monthly['Month_Open']) / monthly['Month_Open'] * 100
monthly['HL_Range_%']        = (monthly['Month_High']  - monthly['Month_Low'])   / monthly['Month_Open'] * 100
monthly['Body_Ratio']        = abs(monthly['Month_Close'] - monthly['Month_Open']) / (monthly['Month_High'] - monthly['Month_Low'])
monthly['Upper_Shadow_%']    = (monthly['Month_High'] - monthly[['Month_Open','Month_Close']].max(axis=1)) / monthly['Month_Open'] * 100
monthly['Lower_Shadow_%']    = (monthly[['Month_Open','Month_Close']].min(axis=1) - monthly['Month_Low'])  / monthly['Month_Open'] * 100
monthly['Recovery_Rate_%'] = (monthly['Month_Close'] - monthly['Month_Low']) / (monthly['Month_High'] - monthly['Month_Low']) * 100

In [37]:
monthly.info()

<class 'pandas.core.frame.DataFrame'>
Index: 308 entries, 0 to 326
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Month             308 non-null    datetime64[ns]
 1   Month_Open        308 non-null    float64       
 2   Month_Close       308 non-null    float64       
 3   Month_High        308 non-null    float64       
 4   Month_Low         308 non-null    float64       
 5   Avg_Volume        308 non-null    float64       
 6   Monthly_Return_%  308 non-null    float64       
 7   HL_Range_%        308 non-null    float64       
 8   Body_Ratio        308 non-null    float64       
 9   Upper_Shadow_%    308 non-null    float64       
 10  Lower_Shadow_%    308 non-null    float64       
 11  Recovery_Rate_%   308 non-null    float64       
dtypes: datetime64[ns](1), float64(11)
memory usage: 31.3 KB


In [38]:
# Target variable
monthly['Next_Month_Return_%'] = monthly['Monthly_Return_%'].shift(-1)
monthly['Next_Month_Direction'] = (monthly['Next_Month_Return_%'] > 0).astype(int)

# Drop last row (NaN target after shift)
monthly = monthly.dropna(subset=['Next_Month_Return_%'])

print(monthly[['Month', 'Monthly_Return_%', 'Next_Month_Return_%', 'Next_Month_Direction']].tail())
print(f"\nShape: {monthly.shape}")
print(f"Direction balance: {monthly['Next_Month_Direction'].value_counts().to_dict()}")

         Month  Monthly_Return_%  Next_Month_Return_%  Next_Month_Direction
321 2025-10-01          4.474103             1.969501                     1
322 2025-11-01          1.969501            -0.745281                     0
323 2025-12-01         -0.745281            -3.257711                     0
324 2026-01-01         -3.257711             1.541146                     1
325 2026-02-01          1.541146             0.837208                     1

Shape: (307, 14)
Direction balance: {1: 176, 0: 131}


In [39]:
monthly.info()

<class 'pandas.core.frame.DataFrame'>
Index: 307 entries, 0 to 325
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Month                 307 non-null    datetime64[ns]
 1   Month_Open            307 non-null    float64       
 2   Month_Close           307 non-null    float64       
 3   Month_High            307 non-null    float64       
 4   Month_Low             307 non-null    float64       
 5   Avg_Volume            307 non-null    float64       
 6   Monthly_Return_%      307 non-null    float64       
 7   HL_Range_%            307 non-null    float64       
 8   Body_Ratio            307 non-null    float64       
 9   Upper_Shadow_%        307 non-null    float64       
 10  Lower_Shadow_%        307 non-null    float64       
 11  Recovery_Rate_%       307 non-null    float64       
 12  Next_Month_Return_%   307 non-null    float64       
 13  Next_Month_Direction  307